# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shashank007-ux/Week-1-Run-the-Starter-Notebooks/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. Method choice and why

I will start with logistic regression because the target is a yes/no observed decline label and the model is readable. I will also compare a shallow decision tree and a random forest. Their predicted probabilities become ranking scores, which fits the Refresh / Content Opportunity Scoring lane. I will use the random forest for the final error analysis because it earns the extra complexity with the strongest overall ranking quality, while still comparing it against the simpler models.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42


def find_repo_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(labels)[order].mean())


ROOT = find_repo_root()
df = pd.read_csv(ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id").reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)

print(f"Loaded {len(df):,} visible content items")
print(f"Observed decline-label base rate: {df['is_declining_label'].mean():.3f}")

Loaded 30,000 visible content items
Observed decline-label base rate: 0.542


## 2. Split design

I will hold out whole clients when possible. Pages from the same client can share traffic conditions and content practices, so putting pages from one client in both train and test can make performance look too strong. The split uses a fixed seed so it can be reproduced. If a client holdout cannot contain both label classes, the code falls back to a stratified row split and reports that limitation.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
target = df["is_declining_label"].astype(int)
all_indices = np.arange(len(df))
clients = df["client_id"].fillna("unknown").astype(str)
unique_clients = clients.drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
test_clients = set(shuffled_clients[:test_client_count])
test_mask = clients.isin(test_clients).to_numpy()
train_indices = all_indices[~test_mask]
test_indices = all_indices[test_mask]

if (target.iloc[train_indices].nunique() < 2 or target.iloc[test_indices].nunique() < 2):
    train_indices, test_indices = train_test_split(
        all_indices, test_size=0.20, random_state=RANDOM_STATE, stratify=target
    )
    split_strategy = "stratified_row_holdout_fallback"
else:
    split_strategy = "client_holdout"

print("Split:", split_strategy)
print("Train rows:", len(train_indices), "Test rows:", len(test_indices))
print("Train clients:", df.iloc[train_indices]["client_id"].nunique())
print("Test clients:", df.iloc[test_indices]["client_id"].nunique())

Split: client_holdout
Train rows: 27675 Test rows: 2325
Train clients: 26
Test clients: 6


## 3. Train + compare vs my baseline

The baseline and every learned model use the same rows, the same client holdout, and the same ranking metrics. The baseline is a transparent weighted opportunity score. Model probabilities are evaluated as ranking scores using precision@20, precision@50, precision@100, average precision, and ROC-AUC. Precision@K matters most because an editor will review only the top part of the queue.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_columns = [
    "search_volume", "competition", "cpc", "content_type", "main_intent",
    "word_count", "char_count", "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "age_tier_order", "days_since_last_update", "ctr",
    "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
feature_columns = [column for column in feature_columns if column in df.columns]
categorical_columns = [column for column in ["content_type", "main_intent"] if column in feature_columns]
numeric_columns = [column for column in feature_columns if column not in categorical_columns]

numeric_frame = df[numeric_columns].apply(pd.to_numeric, errors="coerce")
numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = df[categorical_columns].fillna("unknown").astype(str)
encoded = pd.get_dummies(categorical_frame, prefix=categorical_columns, dtype=float)
X = pd.concat([numeric_frame.reset_index(drop=True), encoded.reset_index(drop=True)], axis=1)

# Recreate the Week-4-style transparent baseline without using the label.
def percentile_rank(series):
    return series.rank(method="average", pct=True).fillna(0)

visibility = percentile_rank(np.log1p(df["impressions_90d"]))
freshness = percentile_rank(df["days_since_last_update"])
position = df["avg_position"].clip(lower=1, upper=50)
position_opportunity = (1 - (position - 1) / 49) * visibility * (df["avg_position"] > 0)
depth_gap = (1 - percentile_rank(df["word_count"])) * visibility
baseline_scores = (0.40 * visibility + 0.30 * freshness + 0.20 * position_opportunity + 0.10 * depth_gap).clip(0, 1)

models = {
    "logistic_regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25, n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE),
}

train_X, test_X = X.iloc[train_indices], X.iloc[test_indices]
train_y, test_y = target.iloc[train_indices], target.iloc[test_indices]

rows = []
def metric_row(name, scores):
    return {
        "method": name,
        "precision_at_20": precision_at_k(scores, test_y, 20),
        "precision_at_50": precision_at_k(scores, test_y, 50),
        "precision_at_100": precision_at_k(scores, test_y, 100),
        "average_precision": average_precision_score(test_y, scores),
        "roc_auc": roc_auc_score(test_y, scores),
    }

rows.append(metric_row("baseline", baseline_scores.iloc[test_indices].to_numpy()))
trained_models = {}
model_scores = {}
for name, model in models.items():
    model.fit(train_X, train_y)
    scores = model.predict_proba(test_X)[:, 1]
    trained_models[name] = model
    model_scores[name] = scores
    rows.append(metric_row(name, scores))

comparison = pd.DataFrame(rows)
display(comparison.round(3))
print("Same split:", split_strategy, "| random seed:", RANDOM_STATE)

,method,precision_at_20,precision_at_50,precision_at_100,average_precision,roc_auc
0,baseline,0.15,0.26,0.34,0.470,0.629
1,logistic_regression,0.95,0.84,0.83,0.640,0.719
2,decision_tree,0.95,0.90,0.85,0.706,0.845
3,random_forest,0.90,0.92,0.90,0.801,0.893


Same split: client_holdout | random seed: 42


## 4. Errors and interpretation

I will inspect false positives and false negatives instead of treating one score as the whole result. False positives are pages ranked as likely declining but not labeled declining; false negatives are declining pages the model missed. Feature importance is directional, not causal: a high-importance feature is useful for prediction, not proof that changing it will recover traffic.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
chosen_name = "random_forest"
chosen_model = trained_models[chosen_name]
chosen_scores = model_scores[chosen_name]

error_frame = df.iloc[test_indices][
    ["content_id", "client_id", "is_declining_label", "trend_direction", "impressions_90d", "avg_position", "ctr", "days_since_last_update"]
].copy()
error_frame["predicted_probability"] = chosen_scores
error_frame["predicted_label"] = (chosen_scores >= 0.5).astype(int)
error_frame["error_type"] = np.select(
    [
        (error_frame["predicted_label"] == 1) & (error_frame["is_declining_label"] == 0),
        (error_frame["predicted_label"] == 0) & (error_frame["is_declining_label"] == 1),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

print("Final model for interpretation:", chosen_name)
print("Confusion matrix [rows=actual, columns=predicted]:")
print(confusion_matrix(test_y, (chosen_scores >= 0.5).astype(int)))
print("False positives:", int((error_frame["error_type"] == "false_positive").sum()))
print("False negatives:", int((error_frame["error_type"] == "false_negative").sum()))
print("\nThree concrete difficult cases:")
display(error_frame[error_frame["error_type"] != "correct"].sort_values("predicted_probability").head(3))

# Permutation importance on held-out data gives a model-agnostic importance check.
importance = permutation_importance(
    chosen_model, test_X, test_y, scoring="average_precision", n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1
)
importance_frame = pd.DataFrame({"feature": X.columns, "importance": importance.importances_mean})
print("\nTop three predictive features:")
display(importance_frame.sort_values("importance", ascending=False).head(3))

assert "trend_direction" not in feature_columns
assert "trend_pct" not in feature_columns
assert "clicks_last_30d" not in feature_columns
assert "sessions_last_30d" not in feature_columns
assert comparison["method"].tolist() == ["baseline", "logistic_regression", "decision_tree", "random_forest"]
print("Leakage checks passed.")

Final model for interpretation: random_forest
Confusion matrix [rows=actual, columns=predicted]:
[[1009  407]
 [  45  864]]
False positives: 407
False negatives: 45

Three concrete difficult cases:


,content_id,client_id,is_declining_label,trend_direction,impressions_90d,avg_position,ctr,days_since_last_update,predicted_probability,predicted_label,error_type
10039,content_fd21cc3cb506,client_d4735e3a26,1,down,10186,3.5,1.97,20,0.340419,0,false_negative
3879,content_34b14c00f80c,client_d4735e3a26,1,down,3,0.0,0.00,20,0.385502,0,false_negative
3914,content_6b4ec59ee781,client_d4735e3a26,1,down,26,24.5,3.85,8,0.392137,0,false_negative



Top three predictive features:


,feature,importance
15,impressions_prev_30d,0.273757
21,ctr,0.031713
6,clicks_90d,0.023669


Leakage checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.